# Quinze ans, deux épargnants · *Fifteen years, two savers*

Notebook compagnon du chapitre **2. Pourquoi un investisseur a besoin de comprendre l'économie : l'intuition de départ** — [lire l'article](https://nmlab.io/ressources/pourquoi-un-investisseur-doit-comprendre-l-economie).
Companion notebook to chapter **2. Why an Investor Needs to Understand the Economy: The Starting Intuition** — [read the article](https://nmlab.io/en/ressources/why-investors-need-to-understand-the-economy).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import numpy as np
from pandas import Series, date_range

def load_paths() -> tuple[Series, Series]:
    """Deux trajectoires de portefeuille simulées sur quinze ans (base 100), 2010-2025.
    L'épargnant immobile reste investi ; l'actif vend au creux de 2020 et revient après
    le rebond, verrouillant son retard. Données simulées à graine fixe — illustration
    stylisée dans l'esprit des études SPIVA et Mind the Gap.
    Two simulated fifteen-year portfolio paths (base 100). The active saver sells at the
    2020 trough and returns after the rebound, locking in his lag. Fixed-seed data."""
    dates = date_range("2010-01-01", "2025-06-01", freq="MS")
    t = np.array([d.year + (d.month - 1) / 12 for d in dates])
    # marché = épargnant immobile : points de contrôle interpolés
    imm_x = [2010, 2013, 2016, 2019, 2020.0, 2020.25, 2020.42, 2021.5, 2022.8, 2024.2, 2025.5]
    imm_y = [100, 124, 150, 184, 205, 221, 176, 238, 201, 270, 298]
    # ondulation douce : pont brownien à graine fixe, ancré à 0 aux extrémités
    rng = np.random.default_rng(11)
    wander = np.cumsum(rng.normal(0, 0.9, len(t)))
    wander = (wander - np.linspace(0, wander[-1], len(t))) * 0.45
    immobile = np.interp(t, imm_x, imm_y) + wander
    # épargnant actif : identique jusqu'à la vente, puis trajectoire propre
    act_x = [2020.42, 2021.1, 2021.7, 2023.0, 2024.2, 2025.5]
    act_y = [176, 176, 193, 163, 169, 180]
    active = immobile.copy()
    sold = t >= 2020.42
    active[sold] = np.interp(t[sold], act_x, act_y) + wander[sold]
    return Series(immobile, index=dates), Series(active, index=dates)

immobile, active = load_paths()


import matplotlib.dates as mdates
from matplotlib.figure import Figure
from pandas import Series

LABELS = {
    "fr": dict(
        title="Quinze ans, deux épargnants",
        sub="Le même marché, deux comportements — la scène qui ouvre le chapitre (stylisée)",
        ylab="portefeuille, base 100",
        immobile="l'épargnant immobile", active="l'épargnant actif",
        mult_imm="\u00d7 3,0", mult_act="\u00d7 1,8",
        sells="sort après la chute", returns="revient après le rebond",
        note="Illustration stylisée (données simulées), dans l'esprit des études SPIVA et Mind the Gap :\n"
             "rester investi bat presque toujours les allers-retours dictés par les convictions du moment."),
    "en": dict(
        title="Fifteen years, two savers",
        sub="The same market, two behaviors — the scene that opens the chapter (stylized)",
        ylab="portfolio, base 100",
        immobile="the motionless saver", active="the active saver",
        mult_imm="\u00d7 3.0", mult_act="\u00d7 1.8",
        sells="sells after the fall", returns="returns after the rebound",
        note="Stylized illustration (simulated data), in the spirit of the SPIVA and Mind the Gap studies:\n"
             "staying invested almost always beats the round-trips dictated by the conviction of the moment."),
}

def build_figure(immobile: Series, active: Series, lang: str) -> Figure:
    """Deux trajectoires base 100 : l'immobile (×3,0) contre l'actif (×1,8)."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1083)
    ax = nm.axes(fig, left=0.088, right=0.93)
    line_act, = ax.plot(active.index, active, color=nm.COLORS["rose"], linewidth=2.9, label=text["active"], zorder=3)
    line_imm, = ax.plot(immobile.index, immobile, color=nm.COLORS["blue"], linewidth=3.1, label=text["immobile"], zorder=4)
    ax.set_ylim(90, 315)
    ax.set_yticks(range(100, 301, 25))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.set_xlim(immobile.index[0], immobile.index[-1])
    ax.xaxis.set_major_locator(mdates.YearLocator(3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.annotate(text["mult_imm"], xy=(immobile.index[-1], immobile.iloc[-1]), xytext=(12, 0),
                textcoords="offset points", ha="left", va="center", fontsize=27,
                fontweight="bold", color=nm.COLORS["blue2"], annotation_clip=False)
    ax.annotate(text["mult_act"], xy=(active.index[-1], active.iloc[-1]), xytext=(12, 0),
                textcoords="offset points", ha="left", va="center", fontsize=27,
                fontweight="bold", color=nm.COLORS["rose"], annotation_clip=False)
    leg = ax.legend(handles=[line_imm, line_act], loc="upper left", fontsize=22, framealpha=1.0,
                    facecolor=nm.COLORS["card"], edgecolor=nm.COLORS["edge"], labelcolor=nm.COLORS["text"],
                    borderpad=0.9, handlelength=1.6, bbox_to_anchor=(0.02, 0.98))
    leg.set_zorder(6)
    sell_x = active.index[(active.index.year == 2020) & (active.index.month == 6)][0]
    ax.annotate(text["sells"], xy=(sell_x, active.loc[sell_x]), xytext=(0, -78),
                textcoords="offset points", ha="center", va="top", fontsize=21,
                color=nm.COLORS["rose"], arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.2))
    ret_x = active.index[(active.index.year == 2021) & (active.index.month == 8)][0]
    ax.annotate(text["returns"], xy=(ret_x, active.loc[ret_x]), xytext=(6, 58),
                textcoords="offset points", ha="center", va="bottom", fontsize=21,
                color=nm.COLORS["rose"], arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.2))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(immobile, active, LANG)